# Customer Churn Prediction and Customer Segmentation System

**Dataset:** Telco Customer Churn  
**Models:** Random Forest Classifier + K-Means Clustering  
**Techniques:** EDA, Class Balancing, Hyperparameter Tuning, Cross-Validation, ROC-AUC, Feature Importance

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)
from sklearn.cluster import KMeans

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('Libraries imported successfully!')

## Step 2: Load Dataset

In [ ]:
df = pd.read_excel('Telco_customer_churn.xlsx')
print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Dataset Info:')
df.info()

In [ ]:
print('Statistical Summary:')
df.describe()

In [ ]:
print('Missing Values per Column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Identify the churn column
churn_col = None
for col in df.columns:
    if 'churn' in col.lower() and 'value' in col.lower():
        churn_col = col
        break
if churn_col is None:
    for col in df.columns:
        if 'churn' in col.lower() and df[col].dtype in ['int64', 'float64']:
            churn_col = col
            break
if churn_col is None:
    for col in df.columns:
        if col.lower() == 'churn':
            churn_col = col
            break
print('Target column:', churn_col)
print('\nChurn Distribution:')
print(df[churn_col].value_counts())

In [ ]:
# --- Churn Distribution ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Exploratory Data Analysis - Customer Churn', fontsize=16, fontweight='bold')

# Plot 1: Churn Distribution
churn_counts = df[churn_col].value_counts()
axes[0,0].pie(churn_counts, labels=['Not Churned (0)', 'Churned (1)'],
              autopct='%1.1f%%', colors=['#66b3ff','#ff6b6b'], startangle=90)
axes[0,0].set_title('Churn Distribution')

# Plot 2: Tenure Distribution
tenure_col = [c for c in df.columns if 'tenure' in c.lower()]
if tenure_col:
    tc = tenure_col[0]
    axes[0,1].hist(df[df[churn_col]==0][tc].dropna(), bins=30, alpha=0.6, label='Not Churned', color='#66b3ff')
    axes[0,1].hist(df[df[churn_col]==1][tc].dropna(), bins=30, alpha=0.6, label='Churned', color='#ff6b6b')
    axes[0,1].set_title('Tenure Distribution by Churn')
    axes[0,1].set_xlabel('Tenure (Months)')
    axes[0,1].set_ylabel('Count')
    axes[0,1].legend()

# Plot 3: Monthly Charges
monthly_col = [c for c in df.columns if 'monthly charge' in c.lower()]
if monthly_col:
    mc = monthly_col[0]
    df.groupby(churn_col)[mc].mean().plot(kind='bar', ax=axes[0,2],
        color=['#66b3ff','#ff6b6b'], rot=0)
    axes[0,2].set_title('Avg Monthly Charges by Churn')
    axes[0,2].set_xlabel('Churn (0=No, 1=Yes)')
    axes[0,2].set_ylabel('Monthly Charges ($)')

# Plot 4: Contract Type
contract_col = [c for c in df.columns if 'contract' in c.lower()]
if contract_col:
    cc = contract_col[0]
    churn_by_contract = df.groupby(cc)[churn_col].mean() * 100
    churn_by_contract.plot(kind='bar', ax=axes[1,0], color='#ff6b6b', rot=45)
    axes[1,0].set_title('Churn Rate by Contract Type (%)')
    axes[1,0].set_ylabel('Churn Rate (%)')

# Plot 5: Payment Method
payment_col = [c for c in df.columns if 'payment' in c.lower()]
if payment_col:
    pc = payment_col[0]
    churn_by_payment = df.groupby(pc)[churn_col].mean() * 100
    churn_by_payment.plot(kind='barh', ax=axes[1,1], color='#66b3ff')
    axes[1,1].set_title('Churn Rate by Payment Method (%)')
    axes[1,1].set_xlabel('Churn Rate (%)')

# Plot 6: Tech Support
tech_col = [c for c in df.columns if 'tech support' in c.lower()]
if tech_col:
    tsc = tech_col[0]
    churn_by_tech = df.groupby(tsc)[churn_col].mean() * 100
    churn_by_tech.plot(kind='bar', ax=axes[1,2], color=['#66b3ff','#ff6b6b','#99ff99'], rot=0)
    axes[1,2].set_title('Churn Rate by Tech Support (%)')
    axes[1,2].set_ylabel('Churn Rate (%)')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved.')

In [ ]:
# Correlation Heatmap
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(numeric_df.corr(), dtype=bool))
sns.heatmap(numeric_df.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5)
plt.title('Correlation Heatmap - Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Data Preprocessing

In [ ]:
# Drop irrelevant columns
drop_cols = []
drop_keywords = ['customerid', 'customer id', 'country', 'state', 'city',
                 'zip code', 'postal', 'latitude', 'longitude',
                 'churn label', 'churn score', 'cltv', 'churn reason']
for col in df.columns:
    if any(kw in col.lower() for kw in drop_keywords):
        drop_cols.append(col)

print('Columns to drop:', drop_cols)
df_clean = df.drop(columns=drop_cols, errors='ignore').copy()
print('Shape after dropping:', df_clean.shape)

In [ ]:
# Handle Total Charges - convert to numeric, fill NaN with 0
total_charges_col = [c for c in df_clean.columns if 'total charge' in c.lower()]
if total_charges_col:
    tc_col = total_charges_col[0]
    df_clean[tc_col] = pd.to_numeric(df_clean[tc_col], errors='coerce')
    missing_before = df_clean[tc_col].isnull().sum()
    df_clean[tc_col] = df_clean[tc_col].fillna(0)
    print(f'Total Charges: {missing_before} missing values filled with 0')

print('\nRemaining missing values:')
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

In [ ]:
# Identify target column in cleaned df
churn_clean = None
for col in df_clean.columns:
    if col.lower() == churn_col.lower():
        churn_clean = col
        break

# Separate features and target
X = df_clean.drop(columns=[churn_clean])
y = df_clean[churn_clean]

# Ensure y is binary numeric
if y.dtype == object:
    y = y.map({'Yes': 1, 'No': 0, 'True': 1, 'False': 0}).fillna(y)
    y = pd.to_numeric(y, errors='coerce').fillna(0).astype(int)

print('Feature shape:', X.shape)
print('Target distribution:\n', y.value_counts())

In [ ]:
# One-Hot Encoding for categorical columns
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print('Categorical columns to encode:', cat_cols)

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print('Shape after encoding:', X_encoded.shape)
X_encoded.head()

## Step 5: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

## Step 6: Initial Random Forest Model

In [ ]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, label='Model'):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)
    print(f'\n=== {label} ===')
    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall   : {rec:.4f}')
    print(f'F1 Score : {f1:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_te, y_pred))
    return model, y_pred, acc, prec, rec, f1

In [ ]:
# Initial model
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base, y_pred_base, acc1, prec1, rec1, f1_1 = evaluate_model(
    rf_base, X_train, X_test, y_train, y_test, 'Base Random Forest'
)

## Step 7: Model Improvement

In [ ]:
# Approach 1: Class weight balanced
rf_balanced = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_balanced, y_pred_bal, acc2, prec2, rec2, f1_2 = evaluate_model(
    rf_balanced, X_train, X_test, y_train, y_test, 'Balanced Class Weight RF'
)

In [ ]:
# Approach 2: Hyperparameter tuning
rf_tuned = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight='balanced', random_state=42
)
rf_tuned, y_pred_tuned, acc3, prec3, rec3, f1_3 = evaluate_model(
    rf_tuned, X_train, X_test, y_train, y_test, 'Tuned RF (n=200, depth=10)'
)

In [ ]:
# Approach 3: Feature Importance Analysis
feature_importances = pd.Series(
    rf_tuned.feature_importances_, index=X_encoded.columns
).sort_values(ascending=False)

plt.figure(figsize=(12, 7))
feature_importances.head(20).plot(kind='barh', color='steelblue')
plt.title('Top 20 Feature Importances (Tuned Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 Features:')
print(feature_importances.head(10))

In [ ]:
# Approach 4: Combination Testing
results = []
combos = [
    (100, 5), (100, 10), (100, 15),
    (200, 5), (200, 10), (200, 15),
    (300, 10), (500, 15)
]
print('Combination Testing (n_estimators, max_depth):')
print(f'{"Trees":>6} {"Depth":>6} {"Accuracy":>10} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('-' * 55)
best_f1 = 0
best_rf = None
best_params = None
for n_est, depth in combos:
    clf = RandomForestClassifier(n_estimators=n_est, max_depth=depth,
                                  class_weight='balanced', random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    y_p = clf.predict(X_test)
    a = accuracy_score(y_test, y_p)
    p = precision_score(y_test, y_p, zero_division=0)
    r = recall_score(y_test, y_p, zero_division=0)
    f = f1_score(y_test, y_p, zero_division=0)
    print(f'{n_est:>6} {depth:>6} {a:>10.4f} {p:>10.4f} {r:>8.4f} {f:>8.4f}')
    results.append({'n_estimators': n_est, 'max_depth': depth,
                    'accuracy': a, 'precision': p, 'recall': r, 'f1': f})
    if f > best_f1:
        best_f1 = f
        best_rf = clf
        best_params = (n_est, depth)

print(f'\nBest combo: n_estimators={best_params[0]}, max_depth={best_params[1]}, F1={best_f1:.4f}')

## Step 8: Confusion Matrix

In [ ]:
y_pred_best = best_rf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted No Churn', 'Predicted Churn'],
            yticklabels=['Actual No Churn', 'Actual Churn'],
            linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix - Best Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives : {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')
print(f'True Positives : {tp}')

## Step 9: Cross Validation

In [ ]:
cv_scores = cross_val_score(best_rf, X_encoded, y, cv=5, scoring='f1', n_jobs=-1)
print('5-Fold Cross Validation F1 Scores:')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'  Mean F1 : {cv_scores.mean():.4f}')
print(f'  Std Dev : {cv_scores.std():.4f}')

plt.figure(figsize=(8, 4))
plt.bar(range(1, 6), cv_scores, color='steelblue', alpha=0.7)
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean F1 = {cv_scores.mean():.4f}')
plt.title('5-Fold Cross Validation F1 Scores', fontsize=13, fontweight='bold')
plt.xlabel('Fold')
plt.ylabel('F1 Score')
plt.legend()
plt.tight_layout()
plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10: ROC-AUC Analysis

In [ ]:
y_prob = best_rf.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC Curve (AUC = {auc_score:.4f})')
plt.plot([0,1], [0,1], color='navy', lw=1, linestyle='--', label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.1, color='darkorange')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Best Random Forest Model', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_auc.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'AUC Score: {auc_score:.4f}')
if auc_score > 0.9: print('Performance: Excellent')
elif auc_score > 0.8: print('Performance: Very Good')
elif auc_score > 0.7: print('Performance: Good')
else: print('Performance: Fair')

## Step 11: Customer Segmentation using K-Means

In [ ]:
# Build segmentation features
seg_cols = []
for col in df_clean.columns:
    if any(kw in col.lower() for kw in ['tenure', 'monthly charge', 'total charge']):
        seg_cols.append(col)

print('Segmentation feature columns:', seg_cols)
seg_df = df_clean[seg_cols].copy()
for c in seg_df.columns:
    seg_df[c] = pd.to_numeric(seg_df[c], errors='coerce')
seg_df = seg_df.fillna(0)

# Add churn probability from best model
seg_df = seg_df.iloc[:len(X_encoded)].copy()
churn_prob = best_rf.predict_proba(X_encoded)[:, 1]
seg_df['Churn Probability'] = churn_prob

print('Segmentation DataFrame shape:', seg_df.shape)
seg_df.describe()

In [ ]:
# Feature Scaling
scaler = StandardScaler()
seg_scaled = scaler.fit_transform(seg_df)
print('Scaling applied (StandardScaler)')

In [ ]:
# Elbow Method
inertias = []
K_range = range(1, 16)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(seg_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(K_range, inertias, marker='o', color='steelblue', linewidth=2, markersize=7)
plt.axvline(x=3, color='red', linestyle='--', label='Optimal K = 3')
plt.title('Elbow Method - Optimal Number of Clusters', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (WCSS)')
plt.legend()
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# K-Means with K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
seg_df['Cluster'] = kmeans.fit_predict(seg_scaled)

print('Cluster Distribution:')
print(seg_df['Cluster'].value_counts().sort_index())

print('\nCluster Profiles (mean values):')
print(seg_df.groupby('Cluster').mean().round(2))

In [ ]:
# Customer Segment Labels based on cluster profiles
cluster_means = seg_df.groupby('Cluster')['Churn Probability'].mean()
sorted_clusters = cluster_means.sort_values().index.tolist()

segment_labels = {}
segment_colors = {}
labels_pool = ['Budget Loyal Customers', 'High Risk New Customers', 'Loyal Premium Customers']
colors_pool = ['#66b3ff', '#ff6b6b', '#99ff99']
for i, cluster_id in enumerate(sorted_clusters):
    segment_labels[cluster_id] = f'Cluster {cluster_id} - {labels_pool[i]}'
    segment_colors[cluster_id] = colors_pool[i]

seg_df['Segment'] = seg_df['Cluster'].map(segment_labels)
print('Cluster Mapping:')
for k, v in segment_labels.items():
    print(f'  Cluster {k} -> {v}')

In [ ]:
# Visualize Clusters
cols = seg_df.columns.tolist()
tenure_like = [c for c in cols if 'tenure' in c.lower()]
monthly_like = [c for c in cols if 'monthly' in c.lower()]

x_col = tenure_like[0] if tenure_like else cols[0]
y_col = monthly_like[0] if monthly_like else cols[1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Customer Segmentation (K-Means, K=3)', fontsize=14, fontweight='bold')

colors = [segment_colors[c] for c in seg_df['Cluster']]
scatter = axes[0].scatter(seg_df[x_col], seg_df[y_col],
                           c=seg_df['Cluster'], cmap='Set1', alpha=0.4, s=20)
axes[0].set_xlabel(x_col)
axes[0].set_ylabel(y_col)
axes[0].set_title('Clusters by Tenure vs Monthly Charges')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Churn Probability by Cluster
seg_df.groupby('Cluster')['Churn Probability'].mean().plot(
    kind='bar', ax=axes[1],
    color=[segment_colors[i] for i in sorted(segment_labels.keys())], rot=0
)
axes[1].set_title('Avg Churn Probability by Cluster')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Avg Churn Probability')

plt.tight_layout()
plt.savefig('customer_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 12: Final Model Comparison Summary

In [ ]:
results_df = pd.DataFrame([
    {'Model': 'Base RF (n=100)',             'Accuracy': acc1, 'Precision': prec1, 'Recall': rec1, 'F1': f1_1},
    {'Model': 'Balanced RF (n=100)',          'Accuracy': acc2, 'Precision': prec2, 'Recall': rec2, 'F1': f1_2},
    {'Model': 'Tuned RF (n=200, depth=10)',   'Accuracy': acc3, 'Precision': prec3, 'Recall': rec3, 'F1': f1_3},
])
print('\n===== Final Model Comparison =====')
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
w = 0.2
bars1 = ax.bar(x - 1.5*w, results_df['Accuracy'],  w, label='Accuracy',  color='#66b3ff')
bars2 = ax.bar(x - 0.5*w, results_df['Precision'], w, label='Precision', color='#ff9999')
bars3 = ax.bar(x + 0.5*w, results_df['Recall'],    w, label='Recall',    color='#99ff99')
bars4 = ax.bar(x + 1.5*w, results_df['F1'],        w, label='F1 Score',  color='#ffcc99')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('\n========== PROJECT SUMMARY ==========')
print(f'Dataset size         : {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Features after encode: {X_encoded.shape[1]}')
print(f'Best RF params       : n_estimators={best_params[0]}, max_depth={best_params[1]}')
print(f'Best F1 Score        : {best_f1:.4f}')
print(f'AUC Score            : {auc_score:.4f}')
print(f'CV Mean F1 (5-fold)  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Customer Segments    : 3 (Budget Loyal, High Risk New, Loyal Premium)')
print('\nKey Findings:')
print('  - Month-to-month contract customers have highest churn rate')
print('  - Electronic check payment users churn more')
print('  - Customers without tech support tend to churn more')
print('  - New customers with high charges are highest risk segment')
print('======================================')